## Setup

In [1]:
import os
import optuna
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sklearn.metrics import f1_score, matthews_corrcoef, average_precision_score
import pandas as pd

from src.py_src import util
from src.py_src.models import Specialist910Model

import warnings
pd.options.mode.copy_on_write = False
warnings.filterwarnings("ignore")

In [2]:
load_dotenv()

slided_df_path = os.path.join(os.getenv("XRAY_SLIDED_PATH"), "xray_slided.parquet")
target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
target_columns = [target_class, target_flux]

buffer_limits = (8.0e-6, 2.0e-5)

df_model_input = util.create_df_model_input_opt(slided_df_path, target_columns, "xl_")

Carregando 68 colunas do arquivo Parquet...


## Preparing Data

In [3]:
specialist_910_pool = df_model_input[df_model_input[target_class] > 2].copy()

train_pct = 0.7
val_pct = (1-train_pct)/2

data = util.prepare_data(
    df_model_input=specialist_910_pool,
    target_class_col=target_class,
    lambda_function=lambda lb: 1 if lb >= 4 else 0,
    train_pct=train_pct,
    val_pct=val_pct,
    target_flux_col=target_flux
)

## Discovery Model

In [4]:
discovery_model = Specialist910Model(
    params={
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_jobs': -1,
        'random_state': 42
    },
    buffer_limits=buffer_limits
)

In [5]:
selected_features = discovery_model.discover_top_features(
    x=data['x']['train'],
    y=data['y']['train'],
    flux_values=data['flux']['train'],
    cumulative_threshold=0.90
)

--- Quick Scan (Discovery Mode) ---
Quick Scan concluído. 40 features selecionadas (de 65).


## Hyperparameter Tuning (Optuna)

In [6]:
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'n_estimators': 1000,
        'random_state': 1502,
        'n_jobs': -1,
        'early_stopping_rounds': 50,
        'device': 'cuda',

        'scale_pos_weight': trial.suggest_float("scale_pos_weight", 1.0, 5.0),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 0.1, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }

    model = Specialist910Model(params=params, buffer_limits=buffer_limits, features_to_keep=selected_features)

    model.fit(
        x=data['x']['train'],
        y=data['y']['train'],
        flux_values=data['flux']['train'],
        eval_set=[(data['x']['val'], data['y']['val'])],
        verbose=False
    )

    y_pred_val = model.predict_proba(data['x']['val'])[:, 1]
    y_pred_class = (y_pred_val >= 0.5).astype(int)

    return matthews_corrcoef(data['y']['val'], y_pred_class)

In [7]:
study = optuna.create_study(direction='maximize')
print("\nIniciando tuning...")
study.optimize(objective, n_trials=50)

print(f"\nBest Score: {study.best_value:.4f}")
best_params = study.best_params

best_params.update({
    'n_estimators': 1000, 'objective': 'binary:logistic',
    'eval_metric': 'logloss', 'random_state': 1502,
    'n_jobs': -1, 'early_stopping_rounds': 50
})

[I 2026-03-18 20:24:55,449] A new study created in memory with name: no-name-b92f3425-de95-4e03-b083-b3c84972b436



Iniciando tuning...


[I 2026-03-18 20:24:56,237] Trial 0 finished with value: 0.2783157776504915 and parameters: {'scale_pos_weight': 3.33704959749282, 'max_depth': 3, 'learning_rate': 0.059867654686485555, 'subsample': 0.8495903854573328, 'colsample_bytree': 0.7229806474181361, 'gamma': 2.4373714808169735, 'min_child_weight': 5}. Best is trial 0 with value: 0.2783157776504915.
[I 2026-03-18 20:24:57,382] Trial 1 finished with value: 0.33895138610472464 and parameters: {'scale_pos_weight': 2.468541367545177, 'max_depth': 7, 'learning_rate': 0.03139449564943395, 'subsample': 0.7590207090490788, 'colsample_bytree': 0.6836282819907287, 'gamma': 4.754348120547804, 'min_child_weight': 7}. Best is trial 1 with value: 0.33895138610472464.
[I 2026-03-18 20:24:58,281] Trial 2 finished with value: 0.3110653510031361 and parameters: {'scale_pos_weight': 2.268079994436102, 'max_depth': 7, 'learning_rate': 0.05846481803480601, 'subsample': 0.8263816873187899, 'colsample_bytree': 0.7427511284706707, 'gamma': 3.927186080


Best Score: 0.4271


In [8]:
final_model = Specialist910Model(params=study.best_params, buffer_limits=buffer_limits, features_to_keep=selected_features)
final_model.fit(
    x=data['x']['train'], y=data['y']['train'],
    flux_values=data['flux']['train']
)

,params,"{'colsample_bytree': 0.7771263936402458, 'gamma': 1.3076925012906546, 'learning_rate': 0.19975138605868253, 'max_depth': 3, ...}"
,buffer_limits,None
,buffer_weight,0.2
,threshold,0.5
,features_to_keep,"['Edec', 'xl_mean_12h', ...]"


## Threshold Tuning

In [9]:
fig = final_model.get_threshold_graph(data['x']['test'], data['y']['test'])
plt.show()

In [10]:
final_model.optimize_threshold(data['x']['test'], data['y']['test'])

Threshold de Equilíbrio (P=R): 0.4597


np.float32(0.459716)

## Results

In [11]:
print(final_model.get_classification_report(
    data['x']['test'], data['y']['test'], target_names=['C', 'MX']
))

              precision    recall  f1-score   support

           C       0.48      0.48      0.48     18946
          MX       0.71      0.71      0.71     33611

    accuracy                           0.63     52557
   macro avg       0.60      0.60      0.60     52557
weighted avg       0.63      0.63      0.63     52557



In [12]:
fig, summary = final_model.analyze_flux_errors(
    data['x']['test'], data['y']['test'],
    flux_values=data['flux']['test'],
    buffer_limits=buffer_limits
)
display(summary)
plt.show()

Outcome,TN (Correct Rejection),FP (False Alarm),FP Rate (%),TP (Hit),FN (Miss),FN Rate (%)
Zone,,,,,,
1. Safe/Low Zone,8257,7186,46.5,0,0,NaN
2. Buffer/Transition Zone,925,2578,73.6,8781,3308,27.4
3. Danger/High Zone,0,0,NaN,15066,6456,30.0


In [13]:
error_report = final_model.analyze_error_distribution(
    x=data['x']['test'],
    y_true=data['y']['test'],
    flux_values=data['flux']['test']
)
display(error_report)

,FN (Miss),FP (False Alarm),FN (Miss) Avg Flux,FP (False Alarm) Avg Flux
SolarClass,,,,
C (1.0 - 9.9),15,9764,9.00e-06,6.08e-06
M (1.0 - 9.9),8339,0,3.32e-05,-
X (> M10),1410,0,3.01e-04,-


In [14]:
x_test = data['x']['test']
y_persistence = ((x_test['count_M_24h'] > 0) | (x_test['count_X_24h'] > 0)).astype(int).values

y_pred = final_model.predict(x_test)

In [15]:
metrics_df = final_model.get_comprehensive_metrics(data['x']['test'], data['y']['test'])
print("--- Métricas Globais (TSS, HSS, MCC, AUCs) ---")
display(metrics_df)

--- Métricas Globais (TSS, HSS, MCC, AUCs) ---


,TSS,HSS,MCC,ROC AUC,PR AUC,F1 Score
0,0.1941,0.1941,0.1941,0.6385,0.746,0.7095


In [ ]:
pr_f1 = final_model.calculate_prss(data['y']['test'], y_pred, y_persistence)
print(f"\nPersistence Relative F1 (PR-F1): {pr_f1:.4f}  | (Acima de 0 significa que o modelo bate o baseline)")

In [ ]:
ac_nc_df = final_model.analyze_ac_nc_performance(x_test, data['y']['test'], y_persistence)
print("\n--- Performance em Transições de Estado (AC) vs Estados Estáveis (NC) ---")
display(ac_nc_df)

## Features Importance

In [ ]:
features_importance = final_model.get_feature_importance()
features_importance

## Export

In [ ]:
specialist_910_dir = os.getenv('SPECIALIST_910_MODELS_PATH')
save_path = os.path.join(specialist_910_dir, '24h/specialist_910_24h_v1.joblib')
final_model.save(save_path)